# XpyriMentor

XpyriMentor implements a more modular way to handle design of experiments. The basic usage is designed to be as close to ProcessOptimizer's Optimizer as possible:

In [1]:
from XpyriMentor import XpyriMentor

space = [[1.0, 100.0], [100, 200], ["cat", "dog", "fish"]]

director = XpyriMentor(space)
print(director)
# Asking for the first parameter set to test
first_suggested_params = director.ask()
print(first_suggested_params)
# Telling the director how well the first parameter set performed
director.tell(first_suggested_params[0], 0.5)
# Asking for the next parameter set to test
second_suggested_params = director.ask()
print(second_suggested_params)
# Telling the director how well the second parameter set performed
director.tell(first_suggested_params[0], 0.5)
# Asking for the next two parameter sets to test
third_and_fourth_suggested_params = director.ask(n=2)
print(third_and_fourth_suggested_params)

XpyriMentor with a SequentialStrategizer suggestor
[[90.10000000000001 170 'dog']]
[[50.5 110 'fish']]
[[70.3 130 'cat']
 [30.7 150 'cat']]


However, we can also make more complicated suggestors. As an example, let's assume we have
a total experiment budget of 30 points. We would like to start with a 7 point Latin
Hypercube sampling, then split the next 20 between an exploring Optimizer (80% likelyhood,
`xi = 10`) and an exploiting Optimizer (20% likelyhood, `xi = 0.00001`), and use the last
experiments with confirming the result by using an exploiting Optimizer ( `xi = 0.00001`).
We set the budget of that last suggestor to be infinite, so we wil keep using that even if
we move beyond our 30 point experiment budget.

In [2]:
suggestor_definition = {
    "suggestor_name": "Sequential",
    "suggestors": [
        {"suggestor_budget": 7, "suggestor_name": "LHS"},
        {
            "suggestor_budget": 20,
            "suggestor_name": "Random",
            "suggestors": [
                {
                    "suggestor_usage_ratio": 80,
                    "suggestor_name": "PO",
                    "acq_func_kwargs": {"xi": 10},
                },
                {
                    "suggestor_usage_ratio": 20,
                    "suggestor_name": "PO",
                    "acq_func_kwargs": {"xi": 0.00001},
                },
            ],
        },
        {
            "suggestor_budget": float("inf"),
            "suggestor_name": "PO",
            "acq_func_kwargs": {"xi": 0.00001}
        },
    ]
}
director = XpyriMentor(space, suggestor_definition)
intial_suggestions = director.ask(5)
director.tell(intial_suggestions, [0.5, 0.6, -0.7, 0.8, 0.9])
print(director.ask(10))

[[78.78571428571428 150 'dog']
 [8.071428571428571 179 'cat']
 [99.83410766319768 159 'cat']
 [72.08039001045324 175 'cat']
 [12.064470941359177 115 'dog']
 [81.95538975576162 143 'fish']
 [41.13246590377519 184 'cat']
 [47.95058608146117 152 'dog']
 [100.0 200 'dog']
 [78.87276505955953 153 'cat']]


You can also make your own suggestor and mix it with the built-in ones. Just make sure
that your suggestor implements the Suggestor protocol.

Specifically, it has to have an `__init__` method, and a `suggest` method which accepts
the input arguments Xi (list of all tested parameters), Yi (list of the results of
testing the parameters) and `n_asked` (the number of suggestions to make).

Note that a different implementation of `ConstantSuggestor` exists in
`XpyriMentor.suggestors`, but we are defining our own here for demonstration.

In [3]:
import numpy as np
from XpyriMentor.suggestors import Suggestor
from ProcessOptimizer.space import Space, space_factory

class ConstantSuggestor():
    def __init__(self, space: Space, constant: int = 0.5):
        self.space = space
        self.constant = constant

    def suggest(self, Xi: list[list], Yi: list, n_asked: int = 1) -> np.ndarray:
        return self.space.sample([[self.constant] * len(self.space)]*n_asked)

print(f"ConstantSuggestor is a Suggestor: {issubclass(ConstantSuggestor, Suggestor)}")

space = space_factory([[1.0, 100.0], [100, 200], ["cat", "dog", "fish"]])

middle_suggestor = ConstantSuggestor(space)
print(f"middle point: {middle_suggestor.suggest([], [])}")

corner_suggestor = ConstantSuggestor(space, constant=0.9)
print(f"corner point: {corner_suggestor.suggest([], [])}")

suggestor_definition = {
    "suggestor_name": "Sequential",
    "suggestors": [
        {"suggestor_budget": 3, "suggestor_name": "LHS"},
        {"suggestor_budget": 20, "suggestor_name": "Random", "suggestors": [
            {"suggestor_usage_ratio": 30, "suggestor": middle_suggestor},
            {"suggestor_usage_ratio": 70, "suggestor": corner_suggestor},
        ]},
    ]
}

director = XpyriMentor(space, suggestor_definition)
print(director.ask(10))

ConstantSuggestor is a Suggestor: True
middle point: [[50.5 150 'dog']]
corner point: [[90.10000000000001 190 'fish']]
[[83.5 116 'dog']
 [50.5 184 'fish']
 [17.5 150 'cat']
 [50.5 150 'dog']
 [50.5 150 'dog']
 [90.10000000000001 190 'fish']
 [90.10000000000001 190 'fish']
 [90.10000000000001 190 'fish']
 [90.10000000000001 190 'fish']
 [90.10000000000001 190 'fish']]


Sequential Strategizer has the option of using default suggestors for any suggestor.

The default first suggestor is a Latin Hypercube Sampling suggestor with as many points
as the budget for that suggestor.

The default suggestor for all other suggestors is a ProcessOptimizer Optimizer suggestor
with no initial points.



In [4]:
space = space_factory([[1.0, 100.0], [100, 200], ["cat", "dog", "fish"]])
suggestor_definition = {
    "suggestor_name": "Sequential",
    "suggestors":[
        {"suggestor_budget": 3, "suggestor_name": "Default"},
        {"suggestor_budget": 20, "suggestor_name": "Default"},
    ]
}
director = XpyriMentor(space, suggestor_definition)
print(f"Director is an {director}.")
print(f"Its suggestor is a {director.suggestor}.")
print(f"More specifically, it has a {director.suggestor.suggestors[0][1]} as initial suggestor and a {director.suggestor.suggestors[1][1]} as ultimate suggestor.")

Director is an XpyriMentor with a SequentialStrategizer suggestor.
Its suggestor is a Sequential Strategizer with suggestors: LHSSuggestor, POSuggestor.
More specifically, it has a Latin Hypercube Suggestor with 3 points as initial suggestor and a ProcessOptimizer Suggestor as ultimate suggestor.
